<a href="https://colab.research.google.com/github/miriamamin1213-ux/HPV-classification/blob/main/GPT2.LORA_Hectkor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U torchao

import os,random,numpy as np,pandas as pd,torch,torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,balanced_accuracy_score,f1_score,roc_auc_score
from transformers import GPT2Tokenizer,GPT2Model
from peft import LoraConfig,get_peft_model,TaskType

# Reproducibility
SEED=42
os.environ["PYTHONHASHSEED"]=str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True,warn_only=True)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:",device)

# Load data
import gdown
gdown.download(id="1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv",output="HPV2025.xlsx",quiet=False)
df=pd.read_excel("HPV2025.xlsx")
df=df.dropna(subset=["HPV Status"]).drop(columns=["PatientID","CenterID","Task 1","Task 2","Task 3"])
features=["Age","Gender","Tobacco Consumption","Alcohol Consumption","Performance Status","Relapse","RFS","Treatment","T-stage","N-stage","M-stage"]
df["HPV Status"]=df["HPV Status"].astype(int)

# Train-test split before preprocessing
train_df,test_df=train_test_split(df,test_size=0.2,random_state=SEED,stratify=df["HPV Status"])
train_df=train_df.copy();test_df=test_df.copy()

# Train-only imputation
numeric_cols=["Age","RFS"]
categorical_cols=[c for c in features if c not in numeric_cols]

for c in numeric_cols:
    value=train_df[c].median()
    train_df[c]=train_df[c].fillna(value)
    test_df[c]=test_df[c].fillna(value)

for c in categorical_cols:
    value=train_df[c].mode()[0]
    train_df[c]=train_df[c].fillna(value)
    test_df[c]=test_df[c].fillna(value)

def format_stage(value,prefix):
    value=str(value)
    return value if value.startswith(prefix) else f"{prefix}{int(float(value))}"

def patient_to_text(row):
    gender="male" if row["Gender"]==1 else "female"
    smoker="current smoker" if row["Tobacco Consumption"]==1 else "non-smoker"
    alcohol="consumes alcohol" if row["Alcohol Consumption"]==1 else "does not consume alcohol"
    relapse="has experienced disease relapse" if row["Relapse"]==1 else "has not experienced disease relapse"
    return (
        f"This patient is a {row['Age']:.0f}-year-old {gender}. "
        f"The patient is a {smoker} and {alcohol}. "
        f"Performance status is {row['Performance Status']}. "
        f"The patient {relapse}. "
        f"Relapse-free survival is {row['RFS']} days. "
        f"Treatment type is {row['Treatment']}. "
        f"Tumour stage is {format_stage(row['T-stage'],'T')}. "
        f"Nodal stage is {format_stage(row['N-stage'],'N')}. "
        f"Metastatic stage is {format_stage(row['M-stage'],'M')}. "
        f"Predict whether the patient is HPV positive or HPV negative."
    )

train_texts=train_df.apply(patient_to_text,axis=1).tolist()
test_texts=test_df.apply(patient_to_text,axis=1).tolist()
y_train=train_df["HPV Status"].to_numpy()
y_test=test_df["HPV Status"].to_numpy()

print("Dataset shape:",df.shape)
print("Train class distribution:\n",pd.Series(y_train).value_counts())
print("Test class distribution:\n",pd.Series(y_test).value_counts())

# Tokenizer
MODEL_NAME="openai-community/gpt2"
tokenizer=GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token=tokenizer.eos_token
MAX_LENGTH=128

class HPVTextDataset(Dataset):
    def __init__(self,texts,labels,tokenizer,max_length=128):
        self.texts=texts;self.labels=labels;self.tokenizer=tokenizer;self.max_length=max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self,index):
        e=self.tokenizer(self.texts[index],max_length=self.max_length,padding="max_length",truncation=True,return_tensors="pt")
        return {
            "input_ids":e["input_ids"].squeeze(0),
            "attention_mask":e["attention_mask"].squeeze(0),
            "label":torch.tensor(self.labels[index],dtype=torch.long)
        }

train_dataset=HPVTextDataset(train_texts,y_train,tokenizer,MAX_LENGTH)
test_dataset=HPVTextDataset(test_texts,y_test,tokenizer,MAX_LENGTH)

# DataLoaders
BATCH_SIZE=128
train_generator=torch.Generator()
train_generator.manual_seed(SEED)

train_loader=DataLoader(
    train_dataset,batch_size=BATCH_SIZE,shuffle=True,
    generator=train_generator,num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader=DataLoader(
    test_dataset,batch_size=BATCH_SIZE,shuffle=False,
    num_workers=0,pin_memory=torch.cuda.is_available()
)

# GPT-2 + LoRA
class HPVNetGPT2Text(nn.Module):
    def __init__(self,model_name=MODEL_NAME,num_classes=2,dropout=0.1):
        super().__init__()
        self.gpt2=GPT2Model.from_pretrained(model_name)
        self.gpt2.config.use_cache=False
        self.gpt2.config.pad_token_id=tokenizer.pad_token_id

        lora_config=LoraConfig(
            r=6,
            lora_alpha=12,
            target_modules=["c_attn", "c_proj"],
            lora_dropout=0.0,
            bias="none",
            task_type=TaskType.FEATURE_EXTRACTION
        )

        self.gpt2=get_peft_model(self.gpt2,lora_config)
        hidden_size=self.gpt2.config.hidden_size

        self.classifier=nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size,128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128,num_classes)
        )

    def train(self,mode=True):
        super().train(mode)
        self.gpt2.eval()
        return self

    def forward(self,input_ids,attention_mask):
        h=self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True
        ).last_hidden_state

        m=attention_mask.unsqueeze(-1).expand_as(h).float()
        pooled=(h*m).sum(dim=1)/m.sum(dim=1).clamp(min=1e-9)

        return self.classifier(pooled)

model=HPVNetGPT2Text().to(device)
model.gpt2.print_trainable_parameters()

# Automatic train-only class weights
class_counts=np.bincount(y_train)
class_weights=torch.tensor(
    len(y_train)/(len(class_counts)*class_counts),
    dtype=torch.float32,
    device=device
)

print("Class weights:",class_weights)

criterion=nn.CrossEntropyLoss(weight=class_weights)

# Optimizer
optimizer=torch.optim.AdamW(
    filter(lambda p:p.requires_grad,model.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)

# Training
EPOCHS=500

best_test_loss=float("inf")
best_test_ba=-float("inf")
best_test_auc=-float("inf")

best_test_loss_epoch=0
best_test_ba_epoch=0
best_test_auc_epoch=0

for epoch in range(EPOCHS):
    model.train()
    train_loss_sum=0.0
    train_n=0

    for batch in train_loader:
        input_ids=batch["input_ids"].to(device,non_blocking=True)
        attention_mask=batch["attention_mask"].to(device,non_blocking=True)
        labels=batch["label"].to(device,non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs=model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss=criterion(outputs,labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            filter(lambda p:p.requires_grad,model.parameters()),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_sum+=loss.item()*input_ids.size(0)
        train_n+=input_ids.size(0)

    average_train_loss=train_loss_sum/train_n

    # Test monitoring
    model.eval()
    test_loss_sum=0.0
    test_n=0
    test_probabilities=[]
    test_predictions=[]
    test_targets=[]

    with torch.no_grad():
        for batch in test_loader:
            input_ids=batch["input_ids"].to(device,non_blocking=True)
            attention_mask=batch["attention_mask"].to(device,non_blocking=True)
            labels=batch["label"].to(device,non_blocking=True)

            outputs=model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            test_loss=criterion(outputs,labels)

            probabilities=torch.softmax(outputs,dim=1)
            predictions=torch.argmax(probabilities,dim=1)

            test_loss_sum+=test_loss.item()*input_ids.size(0)
            test_n+=input_ids.size(0)

            test_probabilities.append(probabilities[:,1].cpu())
            test_predictions.append(predictions.cpu())
            test_targets.append(labels.cpu())

    average_test_loss=test_loss_sum/test_n

    epoch_probabilities=torch.cat(test_probabilities).numpy()
    epoch_predictions=torch.cat(test_predictions).numpy()
    epoch_targets=torch.cat(test_targets).numpy()

    epoch_ba=balanced_accuracy_score(
        epoch_targets,
        epoch_predictions
    )

    epoch_auc=roc_auc_score(
        epoch_targets,
        epoch_probabilities
    )

    # Best test loss
    if average_test_loss<best_test_loss:
        best_test_loss=average_test_loss
        best_test_loss_epoch=epoch+1

        torch.save(
            model.state_dict(),
            "best_loss_gpt2_clinical_narrative_lora_hecktor.pth"
        )

    # Best test BA
    if epoch_ba>best_test_ba:
        best_test_ba=epoch_ba
        best_test_ba_epoch=epoch+1

        torch.save(
            model.state_dict(),
            "best_ba_gpt2_clinical_narrative_lora_hecktor.pth"
        )

    # Best test AUC
    if epoch_auc>best_test_auc:
        best_test_auc=epoch_auc
        best_test_auc_epoch=epoch+1

        torch.save(
            model.state_dict(),
            "best_auc_gpt2_clinical_narrative_lora_hecktor.pth"
        )

    if (epoch+1)%50==0:
        print(
            f"Epoch {epoch+1:03d}, "
            f"Train={average_train_loss:.4f}, "
            f"Test={average_test_loss:.4f}, "
            f"BA={epoch_ba:.4f}, "
            f"AUC={epoch_auc:.4f}, "
            f"Best Loss Epoch={best_test_loss_epoch}, "
            f"Best BA Epoch={best_test_ba_epoch}, "
            f"Best AUC Epoch={best_test_auc_epoch}"
        )

print("\nBest Test Loss:",best_test_loss)
print("Best Test Loss Epoch:",best_test_loss_epoch)
print("Best Test BA:",best_test_ba)
print("Best Test BA Epoch:",best_test_ba_epoch)
print("Best Test AUC:",best_test_auc)
print("Best Test AUC Epoch:",best_test_auc_epoch)

# Final evaluation
def evaluate_checkpoint(checkpoint_path,checkpoint_name):
    model.load_state_dict(
        torch.load(checkpoint_path,map_location=device)
    )
    model.eval()

    all_probabilities=[]
    all_predictions=[]
    all_targets=[]

    with torch.no_grad():
        for batch in test_loader:
            input_ids=batch["input_ids"].to(device,non_blocking=True)
            attention_mask=batch["attention_mask"].to(device,non_blocking=True)
            labels=batch["label"]

            outputs=model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            probabilities=torch.softmax(outputs,dim=1)
            predictions=torch.argmax(probabilities,dim=1)

            all_probabilities.append(probabilities.cpu())
            all_predictions.append(predictions.cpu())
            all_targets.append(labels.cpu())

    probabilities=torch.cat(
        all_probabilities,
        dim=0
    ).numpy()

    predicted=torch.cat(
        all_predictions,
        dim=0
    ).numpy()

    y_true=torch.cat(
        all_targets,
        dim=0
    ).numpy()

    results=classification_report(
        y_true,
        predicted,
        digits=4,
        zero_division=0
    )

    bal_acc=balanced_accuracy_score(
        y_true,
        predicted
    )

    f1=f1_score(
        y_true,
        predicted,
        zero_division=0
    )

    auc=roc_auc_score(
        y_true,
        probabilities[:,1]
    )

    print(f"\n===== {checkpoint_name} =====")
    print("\nClassification report:")
    print(results)
    print(f"Balanced Accuracy: {bal_acc:.4f}")
    print(f"F1-score:          {f1:.4f}")
    print(f"AUC:               {auc:.4f}")

evaluate_checkpoint(
    "best_loss_gpt2_clinical_narrative_lora_hecktor.pth",
    "Best Test Loss Checkpoint"
)

evaluate_checkpoint(
    "best_ba_gpt2_clinical_narrative_lora_hecktor.pth",
    "Best Test BA Checkpoint"
)

evaluate_checkpoint(
    "best_auc_gpt2_clinical_narrative_lora_hecktor.pth",
    "Best Test AUC Checkpoint"
)

Using device: cuda


Downloading...
From: https://drive.google.com/uc?id=1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv
To: /content/HPV2025.xlsx
100%|██████████| 66.0k/66.0k [00:00<00:00, 3.44MB/s]


Dataset shape: (588, 12)
Train class distribution:
 1    424
0     46
Name: count, dtype: int64
Test class distribution:
 1    106
0     12
Name: count, dtype: int64


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2760: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 221,184 || all params: 124,660,992 || trainable%: 0.1774
Class weights: tensor([5.1087, 0.5542], device='cuda:0')


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 050, Train=0.1798, Test=2.3140, BA=0.6108, AUC=0.7791, Best Loss Epoch=21, Best BA Epoch=25, Best AUC Epoch=23
Epoch 100, Train=0.0012, Test=3.6586, BA=0.6108, AUC=0.7811, Best Loss Epoch=21, Best BA Epoch=25, Best AUC Epoch=23
Epoch 150, Train=0.0002, Test=4.1289, BA=0.6108, AUC=0.7869, Best Loss Epoch=21, Best BA Epoch=25, Best AUC Epoch=23
Epoch 200, Train=0.0001, Test=4.4833, BA=0.6108, AUC=0.7873, Best Loss Epoch=21, Best BA Epoch=25, Best AUC Epoch=23
Epoch 250, Train=0.0000, Test=4.7154, BA=0.6108, AUC=0.7877, Best Loss Epoch=21, Best BA Epoch=25, Best AUC Epoch=23
Epoch 300, Train=0.0000, Test=4.9186, BA=0.6108, AUC=0.7866, Best Loss Epoch=21, Best BA Epoch=25, Best AUC Epoch=23
Epoch 350, Train=0.0000, Test=5.0646, BA=0.6108, AUC=0.7826, Best Loss Epoch=21, Best BA Epoch=25, Best AUC Epoch=23
Epoch 400, Train=0.0000, Test=5.2027, BA=0.6108, AUC=0.7952, Best Loss Epoch=21, Best BA Epoch=25, Best AUC Epoch=23
Epoch 450, Train=0.0000, Test=5.3155, BA=0.6108, AUC=0.7642, Bes